<a href="https://colab.research.google.com/github/chaeeunmin/chaeeunmin-High-school-students-breakfast-skipping-rate-and-obesity-rate-relationship/blob/main/12_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12주차: 민감도분석
주 분석(11주차 모형 2)을 재현한 뒤, 가정을 하나씩만 바꿔 다시 분석한다. 문서의 모든 숫자는 이 노트북 출력 그대로 옮긴다.

- S-유병률비(필수): 오즈비 대신 포아송 회귀(로그 연결)의 유병률비
- S1 역방향 인과: 체중 조절 중인 학생을 도메인에서 제외
- S2 노출 정의: 결식 기준 주 5회 이상 → 주 3회 이상

In [1]:
# [Cell 1] 불러오기, 변수 만들기 (10·11주차와 동일)
from google.colab import drive
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')
df = df.copy()  # PerformanceWarning 방지

W, H, C = 'W', 'STRATA', 'CLUSTER'

# 고등학생 도메인 (GRADE 4~6)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 노출: 아침식사 결식 (F_BR 1~3 = 최근 7일 중 먹은 날 0~2일 = 주 5회 이상 결식)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notnull(), 0, np.nan))

# 결과: 비만 (BMI >= 25)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notnull(), 0, np.nan))

# 분석 대상 표시변수: 고등학생 & 노출·결과·교란변수(SEX, GRADE, E_SES) 모두 유효
# (행은 지우지 않고 0/1 표시만 한다. 11주차의 in_model12와 같은 정의)
df['in_analysis'] = ((df['domain_hs'] == 1)
                     & df['bf_skip'].notna() & df['obesity'].notna()
                     & df[['SEX', 'GRADE', 'E_SES']].notna().all(axis=1)).astype(int)
df['in_model12'] = df['in_analysis']

print(f"전체 행 수: {len(df):,}")
print(f"분석 대상자(in_analysis=1): {int(df['in_analysis'].sum()):,}명  (11주차 문서: 25,070명)")

Mounted at /content/drive
전체 행 수: 54,170
분석 대상자(in_analysis=1): 25,070명  (11주차 문서: 25,070명)


In [2]:
# [Cell 2] 설계 기반 회귀 함수 (층·집락·가중치 반영, 행 삭제 없음)
# 점추정: 가중 GLM / 분산: 층(STRATA)·집락(CLUSTER) 선형화(샌드위치) 분산
# 도메인 밖 행은 가중치 0으로 두어 층·집락 구조를 그대로 유지한다.
def design_matrix(expo_col, domain):
    # 더미변수는 분석 도메인 안에 있는 범주로만 만든다 (도메인 밖 범주 열이 전부 0이 되어 회귀가 깨지는 것을 방지)
    X = pd.DataFrame({expo_col: df[expo_col]})
    X['SEX_2'] = (df['SEX'] == 2).astype(float).where(df['SEX'].notna())
    for col in ['GRADE', 'E_SES']:
        cats = sorted(df.loc[domain == 1, col].dropna().unique())
        cat = pd.Categorical(df[col], categories=cats)
        dm = pd.get_dummies(cat, prefix=col, drop_first=True).astype(float)
        dm.index = df.index
        dm = dm.where(df[col].isin(cats))
        X = pd.concat([X, dm], axis=1)
    return X

def svy_glm(y, X, domain, family):
    """family: 'logit'(이항, 로짓) 또는 'log'(포아송, 로그 연결). domain은 0/1."""
    d = (domain == 1) & y.notna() & X.notna().all(axis=1)
    w = df[W].where(d, 0.0).to_numpy()
    Xm = sm.add_constant(X.fillna(0).astype(float), has_constant='add')
    ym = y.fillna(0).astype(float)
    fam = sm.families.Binomial() if family == 'logit' else sm.families.Poisson()
    fit = sm.GLM(ym[d], Xm[d], family=fam, freq_weights=df.loc[d, W]).fit()
    beta = fit.params.to_numpy()
    eta = Xm.to_numpy() @ beta
    if family == 'logit':
        mu = 1 / (1 + np.exp(-eta)); vfun = mu * (1 - mu)
    else:
        mu = np.exp(eta); vfun = mu
    score = (w * (ym.to_numpy() - mu))[:, None] * Xm.to_numpy()
    A = Xm.to_numpy().T @ ((w * vfun)[:, None] * Xm.to_numpy())
    s = pd.DataFrame(score, columns=Xm.columns)
    s['h'] = df[H].to_numpy(); s['c'] = df[C].to_numpy()
    t = s.groupby(['h', 'c']).sum().reset_index()
    B = np.zeros((len(beta), len(beta)))
    for _, g in t.groupby('h'):
        if len(g) > 1:
            z = g[list(Xm.columns)].to_numpy()
            z = z - z.mean(axis=0)
            B += len(g) / (len(g) - 1) * (z.T @ z)
    Ainv = np.linalg.inv(A)
    V = Ainv @ B @ Ainv
    se = np.sqrt(np.diag(V))
    res = pd.DataFrame({'coef': beta, 'se': se}, index=Xm.columns)
    res['z'] = res['coef'] / res['se']
    res['p'] = 2 * (1 - stats.norm.cdf(np.abs(res['z'])))
    res['lo'] = res['coef'] - 1.96 * res['se']
    res['hi'] = res['coef'] + 1.96 * res['se']
    return res, int(d.sum())

def effect(res, expo_col):
    r = res.loc[expo_col]
    return dict(est=np.exp(r['coef']), lo=np.exp(r['lo']), hi=np.exp(r['hi']), p=r['p'])

def fmt(e):
    return f"{e['est']:.2f} ({e['lo']:.2f}–{e['hi']:.2f}), p={e['p']:.4f}"

## 주 분석 재현 (11주차 모형 1·2)
11주차 문서의 값과 같은지 먼저 확인한다. 같지 않으면 아래 민감도분석으로 넘어가지 않는다.

In [3]:
# [Cell 3] 주 분석(모형 1, 모형 2) 재실행 -> 11주차와 같은 결과인지 확인
dom = df['in_analysis']
y = df['obesity']
X_main = design_matrix('bf_skip', dom)

res_m1, n_m1 = svy_glm(y, X_main[['bf_skip']], dom, 'logit')
res_m2, n_m2 = svy_glm(y, X_main, dom, 'logit')
e_m1, e_m2 = effect(res_m1, 'bf_skip'), effect(res_m2, 'bf_skip')

print(f"모형 1 (무보정) n={n_m1:,}: OR {fmt(e_m1)}   <- 11주차 문서: 1.07 (1.00–1.15)")
print(f"모형 2 (보정)   n={n_m2:,}: OR {fmt(e_m2)}   <- 11주차 문서: 1.09 (1.01–1.17)")

모형 1 (무보정) n=25,070: OR 1.07 (1.00–1.15), p=0.0557   <- 11주차 문서: 1.07 (1.00–1.15)
모형 2 (보정)   n=25,070: OR 1.09 (1.01–1.17), p=0.0219   <- 11주차 문서: 1.09 (1.01–1.17)


## S-유병률비 (필수)
모형 2와 같은 보정변수, 같은 분석 대상자로 포아송 회귀(로그 연결)를 적합해 유병률비(PR)를 구하고 오즈비와 비교한다.

In [4]:
# [Cell 4] S-유병률비: 포아송 회귀(로그 연결), 모형 2와 같은 보정변수·같은 사람
res_pr, n_pr = svy_glm(y, X_main, dom, 'log')
e_pr = effect(res_pr, 'bf_skip')
print(f"주 분석 OR (모형 2): {fmt(e_m2)}")
print(f"S-유병률비 PR       : {fmt(e_pr)}   (n={n_pr:,})")
print(f"OR - PR = {e_m2['est'] - e_pr['est']:+.3f}  (결과가 흔할수록 OR이 PR보다 1에서 멀어진다)")

주 분석 OR (모형 2): 1.09 (1.01–1.17), p=0.0219
S-유병률비 PR       : 1.07 (1.01–1.13), p=0.0212   (n=25,070)
OR - PR = +0.022  (결과가 흔할수록 OR이 PR보다 1에서 멀어진다)


## S1 역방향 인과: 체중 조절 중인 학생 제외
DAG의 `비만(Y) → 체중 조절 시도(WC_MN)` 경로 점검. 체중 조절 중인 학생을 도메인에서 빼고(`in_analysis`를 0으로 바꾸고) 모형 2를 다시 적합한다. 행은 지우지 않는다.

In [7]:
# [Cell 5] S1: 체중 조절 중인 학생 제외
# ★ 코드북에서 WC_MN의 응답값 의미를 확인한 뒤, '체중 조절을 시도 중'인 응답값을 아래에 적는다.
print(df.loc[df['in_analysis'] == 1, 'WC_MN'].value_counts(dropna=False).sort_index())

WC_TRY_CODES = [1.0, 2.0, 3.0]   # 1: 체중 감소, 2: 체중 증가, 3: 체중 유지 시도 집단 제외
if WC_TRY_CODES is None:
    raise ValueError("WC_MN 코드북을 확인해 WC_TRY_CODES(체중 조절 시도 응답값)를 입력하세요.")

df['in_s1'] = ((df['in_analysis'] == 1) & df['WC_MN'].notna()
               & ~df['WC_MN'].isin(WC_TRY_CODES)).astype(int)
res_s1, n_s1 = svy_glm(y, X_main, df['in_s1'], 'logit')
e_s1 = effect(res_s1, 'bf_skip')
print(f"주 분석 OR: {fmt(e_m2)}  (n={n_m2:,})")
print(f"S1 OR     : {fmt(e_s1)}  (n={n_s1:,}, 제외 {n_m2 - n_s1:,}명)")

WC_MN
1.0    12066
2.0     8675
3.0     1984
4.0     2345
Name: count, dtype: int64
주 분석 OR: 1.09 (1.01–1.17), p=0.0219  (n=25,070)
S1 OR     : 1.07 (0.85–1.35), p=0.5696  (n=2,345, 제외 22,725명)


## S2 노출 정의: 결식 기준 주 3회 이상
결식 기준을 '먹은 날 0~2일(주 5회 이상 결식)'에서 '먹은 날 0-4일(주 3회 이상 결식)'로 바꾼다. 분석 대상은 주 분석과 같은 사람이다.

In [8]:
# [Cell 6] S2: 주 3회 이상 결식 = 먹은 날 0~4일
# F_BR 응답값은 1=0일, 2=1일, 3=2일 순서(주 분석에서 1~3 = 0~2일)이므로 4=3일, 5=4일. 코드북으로 한 번 더 확인할 것.
print(df.loc[df['in_analysis'] == 1, 'F_BR'].value_counts(dropna=False).sort_index())

df['bf_skip_s2'] = np.where(df['F_BR'].isin([1, 2, 3, 4, 5]), 1,
                            np.where(df['F_BR'].notnull(), 0, np.nan))
X_s2 = design_matrix('bf_skip_s2', dom)
res_s2, n_s2 = svy_glm(y, X_s2, dom, 'logit')
e_s2 = effect(res_s2, 'bf_skip_s2')
print(f"주 분석 OR (주 5회 이상): {fmt(e_m2)}  (n={n_m2:,})")
print(f"S2 OR (주 3회 이상)     : {fmt(e_s2)}  (n={n_s2:,})")
print(f"결식군 인원: 주 5회 이상 {int(((dom==1)&(df['bf_skip']==1)).sum()):,}명 -> 주 3회 이상 {int(((dom==1)&(df['bf_skip_s2']==1)).sum()):,}명")

F_BR
1.0    7003
2.0    2016
3.0    2356
4.0    1958
5.0    1466
6.0    2963
7.0    1551
8.0    5757
Name: count, dtype: int64
주 분석 OR (주 5회 이상): 1.09 (1.01–1.17), p=0.0219  (n=25,070)
S2 OR (주 3회 이상)     : 1.14 (1.06–1.23), p=0.0006  (n=25,070)
결식군 인원: 주 5회 이상 11,375명 -> 주 3회 이상 14,799명


## Table 3: 주 분석과 민감도분석 (문서 D-2에 그대로 옮긴다)

In [9]:
# [Cell 7] Table 3 (마크다운 표로 출력)
rows = [
    ("주 분석 (모형 2, OR)",          n_m2, "OR", e_m2),
    ("S-유병률비 (포아송, PR)",        n_pr, "PR", e_pr),
    ("S1 체중 조절 학생 제외 (OR)",    n_s1, "OR", e_s1),
    ("S2 결식 주 3회 이상 (OR)",       n_s2, "OR", e_s2),
]
print("| 분석 | 분석 인원 | 측도 | 추정치 (95% CI) | p값 | 주 분석과의 차이 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for i, (name, n, kind, e) in enumerate(rows):
    diff = "-" if i == 0 else f"{e['est'] - e_m2['est']:+.2f}"
    print(f"| {name} | {n:,} | {kind} | {e['est']:.2f} ({e['lo']:.2f}–{e['hi']:.2f}) | {e['p']:.4f} | {diff} |")

| 분석 | 분석 인원 | 측도 | 추정치 (95% CI) | p값 | 주 분석과의 차이 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| 주 분석 (모형 2, OR) | 25,070 | OR | 1.09 (1.01–1.17) | 0.0219 | - |
| S-유병률비 (포아송, PR) | 25,070 | PR | 1.07 (1.01–1.13) | 0.0212 | -0.02 |
| S1 체중 조절 학생 제외 (OR) | 2,345 | OR | 1.07 (0.85–1.35) | 0.5696 | -0.02 |
| S2 결식 주 3회 이상 (OR) | 25,070 | OR | 1.14 (1.06–1.23) | 0.0006 | +0.05 |
